# Wind Interpolation Plot

Plot RMSE and NLPD against `walks_per_node` for the two sparse GRF variants.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

TARGET_STEPS = [32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
SEEDS = [0, 1, 2]
RESULTS_DIR = Path("results")

csv_candidates = sorted(RESULTS_DIR.glob("wind_interpolation_seed_walk_sweep_*.csv"))
if not csv_candidates:
    raise FileNotFoundError("No sweep results found in ./results")

csv_path = csv_candidates[-1]
df = pd.read_csv(csv_path)
print(f"Loaded: {csv_path}")
df.head()

In [ ]:
plot_df = df[
    (df["model"].isin(["SparseGRF", "SparseDiffusion"]))
    & (df["walks_per_node"].isin(TARGET_STEPS))
    & (df["seed"].isin(SEEDS))
].copy()

stats_rmse = (
    plot_df.groupby(["model", "walks_per_node"])["rmse"]
    .agg(["mean", "std"])
    .reset_index()
)

plt.figure(figsize=(10, 6))
for model in stats_rmse["model"].unique():
    model_df = stats_rmse[stats_rmse["model"] == model]
    plt.plot(model_df["walks_per_node"], model_df["mean"], marker="o", label=model)
    plt.fill_between(
        model_df["walks_per_node"],
        model_df["mean"] - model_df["std"].fillna(0.0),
        model_df["mean"] + model_df["std"].fillna(0.0),
        alpha=0.2,
    )

plt.xscale("log", base=2)
plt.xlabel("Walks per Node")
plt.ylabel("RMSE")
plt.title("Wind Interpolation: RMSE vs Walks per Node")
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.3)
plt.show()

In [ ]:
stats_nlpd = (
    plot_df.groupby(["model", "walks_per_node"])["nlpd"]
    .agg(["mean", "std"])
    .reset_index()
)

plt.figure(figsize=(10, 6))
for model in stats_nlpd["model"].unique():
    model_df = stats_nlpd[stats_nlpd["model"] == model]
    plt.plot(model_df["walks_per_node"], model_df["mean"], marker="s", label=model)
    plt.fill_between(
        model_df["walks_per_node"],
        model_df["mean"] - model_df["std"].fillna(0.0),
        model_df["mean"] + model_df["std"].fillna(0.0),
        alpha=0.2,
    )

plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("Walks per Node")
plt.ylabel("NLPD")
plt.title("Wind Interpolation: NLPD vs Walks per Node")
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.3)
plt.show()